# Client Profile Model

This notebook introduces a simple advisory client profile model. The goal is not to forecast returns, but to capture the client context needed for suitability, monitoring, and explainable advice.

A practical advisory profile should distinguish between:

- **Risk tolerance**: the client's psychological willingness to accept losses.
- **Risk capacity**: the client's financial ability to bear losses.
- **Knowledge and experience**: whether the client can understand product risks.
- **Constraints**: liquidity needs, time horizon, concentration limits, tax considerations, and restrictions.


In [1]:
from pathlib import Path
import sys

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from advice import ProductProfile, suitability_flags
from profiling import ClientProfile, combined_risk_profile, liquidity_ratio, risk_capacity


In [2]:
client = ClientProfile(
    client_id="C-1001",
    age=42,
    annual_income=180_000,
    liquid_net_worth=750_000,
    investment_horizon_years=12,
    liquidity_need_12m=60_000,
    risk_tolerance="medium",
    investment_knowledge="intermediate",
    has_dependents=True,
    restrictions=["no single-stock positions above 10%"],
)

client


ClientProfile(client_id='C-1001', age=42, annual_income=180000, liquid_net_worth=750000, investment_horizon_years=12, liquidity_need_12m=60000, risk_tolerance='medium', investment_knowledge='intermediate', has_dependents=True, restrictions=['no single-stock positions above 10%'])

## Derive Advisory Signals

Suitability checks often use derived signals rather than raw profile fields. The functions below are deliberately simple so the assumptions stay visible.

In [3]:
derived_profile = {
    "liquidity_ratio": round(liquidity_ratio(client), 3),
    "risk_capacity": risk_capacity(client),
    "combined_risk_profile": combined_risk_profile(client),
}

derived_profile


{'liquidity_ratio': 0.08,
 'risk_capacity': 'medium',
 'combined_risk_profile': 'medium'}

## Simple Product Suitability Check

A production advisory system would keep product risk, complexity, concentration, and liquidity attributes in a controlled product master. This example keeps the product data small and explicit.

In [4]:
product = ProductProfile(
    product_id="P-204",
    name="Global Multi-Asset Fund",
    risk_level="medium",
    required_knowledge="basic",
    daily_liquidity=True,
)

product


ProductProfile(product_id='P-204', name='Global Multi-Asset Fund', risk_level='medium', required_knowledge='basic', daily_liquidity=True)

In [5]:
flags = suitability_flags(client, product)

{
    "client_id": client.client_id,
    "product_id": product.product_id,
    "suitable": not flags,
    "flags": flags,
}


{'client_id': 'C-1001', 'product_id': 'P-204', 'suitable': True, 'flags': []}

## Next Extensions

- Add source-of-wealth and source-of-funds fields.
- Separate regulatory knowledge checks from internal advisory policy checks.
- Store profile versions so suitability decisions can be reconstructed later.
- Add evidence fields for documents, questionnaires, and advisor attestations.
